# Tanzania Climate EDA (Task 2)

This notebook performs data profiling, cleaning, and focused EDA for Tanzania (2015-2026) using NASA POWER climate data.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path
from scipy.stats import zscore

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 100)


In [ ]:
country = 'tanzania'
input_path = Path('..') / 'data' / f'{country}.csv'
output_path = Path('..') / 'data' / f'{country}_clean.csv'

df = pd.read_csv(input_path)
df['Country'] = 'Tanzania'
df['Date'] = pd.to_datetime(df['YEAR'] * 1000 + df['DOY'], format='%Y%j', errors='coerce')
df['Month'] = df['Date'].dt.month

df = df.replace(-999, np.nan)
dup_count = int(df.duplicated().sum())
df = df.drop_duplicates().copy()
print('Duplicate rows found and dropped:', dup_count)

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
display(df[numeric_cols].describe())
missing_report = pd.DataFrame({'missing_count': df.isna().sum(), 'missing_pct': (df.isna().mean() * 100).round(2)}).sort_values('missing_pct', ascending=False)
display(missing_report)
display(missing_report[missing_report['missing_pct'] > 5])


In [ ]:
outlier_cols = ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'WS2M', 'WS2M_MAX']
available_outlier_cols = [c for c in outlier_cols if c in df.columns]
z_df = df[available_outlier_cols].apply(lambda s: zscore(s, nan_policy='omit'))
outlier_mask = (z_df.abs() > 3).any(axis=1)
print('Rows flagged as outliers (|Z| > 3):', int(outlier_mask.sum()))

row_missing_pct = df.isna().mean(axis=1)
before = len(df)
df = df[row_missing_pct <= 0.30].copy()
print('Rows dropped (>30% missing):', before - len(df))

weather_fill_cols = [c for c in ['T2M','T2M_MAX','T2M_MIN','T2M_RANGE','PRECTOTCORR','RH2M','WS2M','WS2M_MAX','PS','QV2M'] if c in df.columns]
df = df.sort_values('Date').reset_index(drop=True)
df[weather_fill_cols] = df[weather_fill_cols].ffill()
df.to_csv(output_path, index=False)
print('Clean dataset exported to:', output_path)


In [ ]:
monthly = df.resample('M', on='Date').agg({'T2M': 'mean', 'PRECTOTCORR': 'sum'}).reset_index()
warmest_idx = monthly['T2M'].idxmax()
coolest_idx = monthly['T2M'].idxmin()
rainiest_idx = monthly['PRECTOTCORR'].idxmax()

fig, ax = plt.subplots(2, 1, figsize=(14, 10))
sns.lineplot(data=monthly, x='Date', y='T2M', ax=ax[0], color='firebrick')
ax[0].set_title('Monthly Average T2M (2015-2026)')
ax[0].annotate('Warmest month', (monthly.loc[warmest_idx, 'Date'], monthly.loc[warmest_idx, 'T2M']))
ax[0].annotate('Coolest month', (monthly.loc[coolest_idx, 'Date'], monthly.loc[coolest_idx, 'T2M']))

sns.barplot(data=monthly, x=monthly['Date'].dt.strftime('%Y-%m'), y='PRECTOTCORR', ax=ax[1], color='steelblue')
ax[1].set_title('Monthly Total PRECTOTCORR (2015-2026)')
ax[1].tick_params(axis='x', labelrotation=90)
ax[1].annotate('Peak rainy month', (rainiest_idx, monthly.loc[rainiest_idx, 'PRECTOTCORR']))
plt.tight_layout(); plt.show()


In [ ]:
num_df = df.select_dtypes(include=[np.number]).copy()
corr = num_df.corr(numeric_only=True)
plt.figure(figsize=(12, 8)); sns.heatmap(corr, cmap='coolwarm', center=0); plt.title('Correlation Heatmap'); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=df, x='T2M', y='RH2M', alpha=0.5, ax=ax[0]); ax[0].set_title('T2M vs RH2M')
sns.scatterplot(data=df, x='T2M_RANGE', y='WS2M', alpha=0.5, ax=ax[1]); ax[1].set_title('T2M_RANGE vs WS2M')
plt.tight_layout(); plt.show()

corr_unstacked = corr.where(~np.eye(corr.shape[0], dtype=bool)).abs().unstack().dropna().sort_values(ascending=False)
top3 = corr_unstacked[~corr_unstacked.index.duplicated()].head(3)
display(top3)


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['PRECTOTCORR'].dropna(), bins=40, kde=True, ax=ax[0], color='teal')
ax[0].set_title('Distribution of PRECTOTCORR')

sns.scatterplot(data=df, x='T2M', y='RH2M', size='PRECTOTCORR', sizes=(10, 300), alpha=0.4, ax=ax[1])
ax[1].set_title('Bubble: T2M vs RH2M (size=PRECTOTCORR)')
plt.tight_layout(); plt.show()


## Negotiation-Grade Insight Prompts
- What is changing (trend + baseline + uncertainty)?
- What did it cause (impact indicator from a cited secondary source)?
- What does it demand (policy/finance ask supported by evidence)?
